# 🌿 Cassava Leaf Disease: fine-tuning ConvNeXt-Tiny

This notebook teaches a neural network to spot **4 cassava leaf diseases** (and healthy leaves) from a photo.
It starts from **ConvNeXt-Tiny**, the same ImageNet model used in the
[Image Classifier demo](https://huggingface.co/spaces/Laavanjan/image-classifier), and fine-tunes it on the
[Kaggle cassava photos](https://www.kaggle.com/competitions/cassava-leaf-disease-classification) taken by farmers in Uganda.

**What it does, step by step**

1. Loads the photos (about 21,400) and shows how uneven the classes are.
2. Splits them into train (80 %), validation (10 %) and test (10 %).
3. **Baseline:** freezes the network and trains only the last layer.
4. **Fine-tune:** trains the whole network, with photo augmentations and class weights.
5. Tests both on photos they have never seen: accuracy, macro F1 and a confusion matrix.
6. Shows the 10 mistakes the model was most sure about, with Grad-CAM heatmaps.
7. *(Optional)* Tests on your own photos.
8. Saves the model, the charts and all scores.

**How to run it on Kaggle**

1. Open the [competition page](https://www.kaggle.com/competitions/cassava-leaf-disease-classification) and click
   **Late Submission** (the competition has ended, so there is no **Join** button). A window shows the rules: click
   **I Understand and Accept**. This only accepts the rules, it does not submit anything. Without it you can't use the data.
2. On kaggle.com go to **Code → New Notebook → File → Import Notebook** and upload this file.
3. In the right-hand panel: **Add Input → Competitions → Cassava Leaf Disease Classification**.
4. In **Session options**: set **Accelerator → GPU T4 x2** (only one GPU is used) and turn **Internet on**
   (needed to download the pretrained weights). Both need a phone-verified Kaggle account.
5. Set `QUICK_RUN = True` below and click **Run All**. This small run checks that everything works in a few minutes.
6. Set `QUICK_RUN = False` and click **Save Version → Save & Run All (Commit)**. The full run takes about 1–2 hours
   and keeps going even if you close the browser. When it finishes, the files are in that version's **Output** tab.

## 1. Settings

Change these if you want. Everything else works on its own.

In [ ]:
# ---- CONFIG: change these if you want ----
DATA_DIR = None           # None = find the Kaggle competition data by itself
OUT_DIR = None            # None = /kaggle/working on Kaggle (the Output tab), else the current folder
QUICK_RUN = False         # True = small test run (about 1,000 photos, 1 epoch per step) to check that everything works

IMG_SIZE = 384            # photos are cut to IMG_SIZE x IMG_SIZE pixels; bigger = more detail, but slower
BATCH_SIZE = 32           # photos per training step; use 16 if you see "CUDA out of memory"
BASELINE_EPOCHS = 2       # step 1: train only the last layer
FINETUNE_EPOCHS = 10      # step 2: train the whole network
BASELINE_LR = 1e-3        # learning rate for step 1
FINETUNE_LR = 2e-4        # learning rate for step 2 (smaller, so the ImageNet knowledge is not destroyed)
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1     # helps because some labels in this dataset are wrong
USE_CLASS_WEIGHTS = True  # mistakes on small classes cost more
NUM_WORKERS = 4           # processes that load photos (Kaggle has 4 CPUs); use 0 on Windows
SEED = 42                 # same seed = same split and (almost) the same results
MY_PHOTOS_DIR = None      # optional: a folder with your own photos, see section 9

## 2. Imports and setup

In [ ]:
import json
import math
import os
import random
import time
from glob import glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from torchvision.models import ConvNeXt_Tiny_Weights, convnext_tiny

import warnings
# Mixed precision may skip the first steps while it finds the right scale, and PyTorch then warns that the
# learning-rate schedule moved first. That is normal and harmless, so this one warning is hidden.
warnings.filterwarnings("ignore", message=r"Detected call of `lr_scheduler\.step\(\)`")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"  # mixed precision: much faster on a GPU, same accuracy
torch.backends.cudnn.benchmark = True

if QUICK_RUN:
    BASELINE_EPOCHS, FINETUNE_EPOCHS = 1, 1


def find_data_dir():
    """Look for the folder that has train.csv and train_images/ under /kaggle/input."""
    for csv in glob("/kaggle/input/*/train.csv") + glob("/kaggle/input/*/*/train.csv"):
        folder = os.path.dirname(csv)
        if os.path.isdir(os.path.join(folder, "train_images")):
            return folder
    raise FileNotFoundError("Cassava data not found. Add the competition data (Add Input) or set DATA_DIR.")


DATA_DIR = Path(DATA_DIR or find_data_dir())
OUT_DIR = Path(OUT_DIR or ("/kaggle/working" if os.path.isdir("/kaggle/working") else "."))
OUT_DIR.mkdir(parents=True, exist_ok=True)

gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else "none"
print(f"device: {device} ({gpu_name}) | torch {torch.__version__}")
print(f"data:   {DATA_DIR}\noutput: {OUT_DIR}")
if device.type == "cpu":
    print("⚠️ No GPU found. Turn on a GPU in Session options, or training will be very slow.")

## 3. Look at the data

`train.csv` lists every photo and its label (0–4). The chart shows the big problem with this dataset:
**one disease (CMD) has about 60 % of all photos.** A model that always answers "CMD" would already get about 60 %
accuracy while being useless for the other diseases. So besides accuracy we also use **macro F1**, which gives every
class the same importance.

In [ ]:
df = pd.read_csv(DATA_DIR / "train.csv")
with open(DATA_DIR / "label_num_to_disease_map.json") as f:
    LONG_NAMES = [name for _, name in sorted(json.load(f).items(), key=lambda kv: int(kv[0]))]
CLASSES = [n[n.rfind("(") + 1 : -1] if n.endswith(")") else n for n in LONG_NAMES]  # "... (CMD)" -> "CMD"
NUM_CLASSES = len(CLASSES)

if QUICK_RUN:  # keep about 1,000 photos, with the same class mix
    df = df.groupby("label", group_keys=False).sample(frac=min(1.0, 1000 / len(df)), random_state=SEED)

counts = df["label"].value_counts().reindex(range(NUM_CLASSES), fill_value=0)
print(f"{len(df):,} photos\n")
for i, (short, long) in enumerate(zip(CLASSES, LONG_NAMES)):
    print(f"  {i}  {short:8} {counts[i]:6,}  ({counts[i] / len(df):5.1%})   {long}")

plt.figure(figsize=(7, 3.5))
plt.bar(CLASSES, counts.values, color="#4f46e5")
plt.title("Photos per class: very uneven")
plt.ylabel("photos")
plt.tight_layout()
plt.savefig(OUT_DIR / "class_counts.png", dpi=120)
plt.show()

Four random photos of each class. Notice how different they are: close-ups, whole plants, soil, sky, shadows.

In [ ]:
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(12, 2.7 * NUM_CLASSES))
for c in range(NUM_CLASSES):
    sample = df[df.label == c].sample(min(4, counts[c]), random_state=SEED)
    for ax in axes[c]:
        ax.axis("off")
    for ax, image_id in zip(axes[c], sample.image_id):
        ax.imshow(Image.open(DATA_DIR / "train_images" / image_id))
    axes[c][0].set_title(LONG_NAMES[c], loc="left", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Split into train, validation and test

- **train (80 %)**: the model learns from these.
- **validation (10 %)**: checked after every epoch; we keep the model that scores best here.
- **test (10 %)**: used **only once, at the very end**. This is the honest score you report.

`stratify` keeps the same class mix in all three parts. The split is saved to `split.csv`, so it can be repeated.

In [ ]:
train_df, hold_df = train_test_split(df, test_size=0.2, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(hold_df, test_size=0.5, stratify=hold_df.label, random_state=SEED)

split = pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")])
split.to_csv(OUT_DIR / "split.csv", index=False)
print(split.groupby(["split", "label"]).size().unstack().rename(columns=dict(enumerate(CLASSES))))

## 5. From photos to tensors

**Training photos are changed a little every time they are used**: a random crop, flips and small colour changes.
The model can't just memorise them, so it learns to handle new phone photos.
Validation and test photos are only resized and centre-cropped, the same way the app does it.

In [ ]:
weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
recipe = weights.transforms()
MEAN, STD = list(recipe.mean), list(recipe.std)
RESIZE_SIZE = round(IMG_SIZE * recipe.resize_size[0] / recipe.crop_size[0])  # same ratio as the ImageNet recipe

to_tensor = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    to_tensor,
])
eval_view = T.Compose([T.Resize(RESIZE_SIZE), T.CenterCrop(IMG_SIZE)])  # what the model sees at test time
eval_tf = T.Compose([eval_view, to_tensor])


class PhotoDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths, self.labels, self.transform = list(paths), list(labels), transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = ImageOps.exif_transpose(Image.open(self.paths[i])).convert("RGB")  # exif: phone photos stay upright
        return self.transform(img), self.labels[i]


def make_loader(paths, labels, transform, shuffle=False):
    return DataLoader(
        PhotoDataset(paths, labels, transform),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        drop_last=shuffle,
        num_workers=NUM_WORKERS,
        persistent_workers=NUM_WORKERS > 0,
        pin_memory=device.type == "cuda",
    )


def cassava_paths(frame):
    return [DATA_DIR / "train_images" / image_id for image_id in frame.image_id]


train_loader = make_loader(cassava_paths(train_df), train_df.label, train_tf, shuffle=True)
val_loader = make_loader(cassava_paths(val_df), val_df.label, eval_tf)
test_loader = make_loader(cassava_paths(test_df), test_df.label, eval_tf)
print(f"train {len(train_df):,} | val {len(val_df):,} | test {len(test_df):,} photos")
print(f"input: resize to {RESIZE_SIZE}, crop {IMG_SIZE}x{IMG_SIZE}")

## 6. The model and the loss

We load ConvNeXt-Tiny with its ImageNet weights and **replace its last layer**: 1,000 ImageNet classes → 5 cassava classes.

The loss (how wrong the model is) is cross-entropy with two extras:

- **Class weights**: a mistake on a small class costs more, so the model can't just ignore it.
  We use the square root of "1 ÷ class size", so the small classes get help but not too much.
- **Label smoothing**: the model is taught to be about 90 % sure instead of 100 %. This helps because
  some labels in this dataset are wrong.

In [ ]:
model = convnext_tiny(weights=weights)
model.classifier[2] = nn.Linear(model.classifier[2].in_features, NUM_CLASSES)
model = model.to(device)

train_counts = np.bincount(train_df.label, minlength=NUM_CLASSES)
if USE_CLASS_WEIGHTS:
    cw = np.sqrt(train_counts.sum() / np.maximum(train_counts, 1))
    cw = cw / cw.mean()
else:
    cw = np.ones(NUM_CLASSES)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=device),
                                label_smoothing=LABEL_SMOOTHING)
print("class weights:", {c: round(float(w), 2) for c, w in zip(CLASSES, cw)})
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")

## 7. Training helpers

- `evaluate` runs the model on a set of photos and returns its probabilities.
- `train` runs a number of epochs (one epoch = every training photo once). After each epoch it checks the
  validation set and saves the model **only if it is the best so far**.
- The learning rate warms up at the start, then slowly goes down to zero (a "cosine schedule").

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels, total_loss = [], [], 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(x).float()
        total_loss += criterion(logits, y).item() * len(y)
        all_probs.append(logits.softmax(1).cpu())
        all_labels.append(y.cpu())
    labels = torch.cat(all_labels).numpy()
    return torch.cat(all_probs).numpy(), labels, total_loss / len(labels)


def scores(labels, pred):
    return {
        "accuracy": float(accuracy_score(labels, pred)),
        "macro_f1": float(f1_score(labels, pred, average="macro", labels=range(NUM_CLASSES), zero_division=0)),
    }


def cosine_with_warmup(optimizer, total_steps, warmup_steps):
    def factor(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


history = []


def train(model, params, epochs, lr, phase, save_as):
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
    total_steps = epochs * len(train_loader)
    scheduler = cosine_with_warmup(optimizer, total_steps, warmup_steps=max(1, total_steps // 20))
    scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)
    best_f1 = -1.0
    for epoch in range(1, epochs + 1):
        model.train()
        start, total_loss, seen = time.time(), 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=USE_AMP):
                loss = criterion(model(x).float(), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item() * len(y)
            seen += len(y)

        probs, labels, val_loss = evaluate(model, val_loader)
        s = scores(labels, probs.argmax(1))
        row = {"phase": phase, "epoch": epoch, "train_loss": total_loss / seen, "val_loss": val_loss,
               "val_accuracy": s["accuracy"], "val_macro_f1": s["macro_f1"], "minutes": (time.time() - start) / 60}
        history.append(row)
        note = ""
        if s["macro_f1"] > best_f1:
            best_f1 = s["macro_f1"]
            torch.save(model.state_dict(), OUT_DIR / save_as)
            note = "  ✓ best so far, saved"
        print(f"[{phase}] epoch {epoch}/{epochs}  train loss {row['train_loss']:.3f}  val loss {val_loss:.3f}  "
              f"val acc {s['accuracy']:.3f}  val macro F1 {s['macro_f1']:.3f}  ({row['minutes']:.1f} min){note}")

    model.load_state_dict(torch.load(OUT_DIR / save_as, map_location=device))  # go back to the best epoch
    return best_f1

## 8a. Step 1, the baseline: train only the last layer

All layers except the last stay **frozen**, exactly as they were trained on ImageNet. Only the new last layer learns.
This is quick and gives a score to beat.

In [ ]:
for p in model.features.parameters():
    p.requires_grad = False
best = train(model, model.classifier.parameters(), BASELINE_EPOCHS, BASELINE_LR, "baseline", "baseline.pt")
print(f"\nBaseline: best validation macro F1 = {best:.3f}")

## 8b. Step 2: fine-tune the whole network

Now **every layer is allowed to learn**, with a smaller learning rate. We start from the baseline's trained last
layer instead of a random one, which usually works better. The model with the best validation macro F1 is kept.

In [ ]:
for p in model.parameters():
    p.requires_grad = True
best = train(model, model.parameters(), FINETUNE_EPOCHS, FINETUNE_LR, "finetune", "finetuned.pt")
print(f"\nFine-tuned: best validation macro F1 = {best:.3f}")

hist = pd.DataFrame(history)
hist.to_csv(OUT_DIR / "history.csv", index=False)
x = np.arange(1, len(hist) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(x, hist.train_loss, "o-", label="train")
ax1.plot(x, hist.val_loss, "o-", label="validation")
ax1.set(title="Loss (lower is better)")
ax2.plot(x, hist.val_accuracy, "o-", label="accuracy")
ax2.plot(x, hist.val_macro_f1, "o-", label="macro F1")
ax2.set(title="Validation score (higher is better)")
for ax in (ax1, ax2):
    ax.axvline(BASELINE_EPOCHS + 0.5, color="grey", ls="--")
    ax.set_xlabel("epoch (left of the dashed line: baseline, right: fine-tune)")
    ax.set_xticks(x)
    ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "training_curves.png", dpi=120)
plt.show()

## 8c. The final test: the honest score

Now, for the first and only time, we use the test photos. Both models are tested, so you can show
"baseline → fine-tuned" in your README. A model that always gives the biggest class is added as a floor.

- **Accuracy**: the share of photos it got right.
- **Macro F1**: the F1 score of each class, averaged. Every class counts the same, so one weak class pulls it down.
- **Confusion matrix**: each row is the true class, each column is what the model said. The diagonal is correct,
  everything else is a mistake. Each row adds up to 1.

In [ ]:
results = {}
majority = int(train_counts.argmax())
for name, file in [("baseline", "baseline.pt"), ("finetuned", "finetuned.pt")]:
    model.load_state_dict(torch.load(OUT_DIR / file, map_location=device))
    test_probs, test_labels, _ = evaluate(model, test_loader)
    results[name] = scores(test_labels, test_probs.argmax(1))
results = {"always_" + CLASSES[majority]: scores(test_labels, np.full_like(test_labels, majority)), **results}
test_pred = test_probs.argmax(1)  # the model now holds the fine-tuned weights again

print(f"{'TEST SET':24}{'accuracy':>10}{'macro F1':>10}")
for name, s in results.items():
    print(f"{name:24}{s['accuracy']:>10.3f}{s['macro_f1']:>10.3f}")
print("\nFine-tuned model, per class:\n")
print(classification_report(test_labels, test_pred, labels=range(NUM_CLASSES), target_names=CLASSES,
                            digits=3, zero_division=0))

cm = confusion_matrix(test_labels, test_pred, labels=range(NUM_CLASSES))
cm_share = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm_share, cmap="Blues", vmin=0, vmax=1)
ax.set(xticks=range(NUM_CLASSES), yticks=range(NUM_CLASSES), xticklabels=CLASSES, yticklabels=CLASSES,
       xlabel="model said", ylabel="true class", title="Confusion matrix (test set)")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f"{cm_share[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", fontsize=9,
                color="white" if cm_share[i, j] > 0.5 else "black")
fig.colorbar(im, ax=ax, label="share of the row")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix.png", dpi=120)
plt.show()

## 8d. The mistakes it was most sure about

These are the 10 test photos where the model was **wrong but very confident**. They teach you the most.
For each one, look at the photo and the heatmap and ask:

- Is the **label** wrong? This dataset has many wrong labels.
- Does the leaf show **two diseases** at once?
- Is the photo blurry or dark, or does it show mostly soil or sky?
- Did the heatmap look at the **leaf**, or at the background?

Then copy the table printed below into your README and fill in the **my reason** column.

In [ ]:
def grad_cam(model, x, class_idx):
    """Grad-CAM on the last ConvNeXt stage: the same method as the live demo (classifier.py)."""
    store = {}

    def hook(_module, _inputs, out):
        store["act"] = out.detach()
        out.register_hook(lambda grad: store.__setitem__("grad", grad.detach()))

    handle = model.features[-1].register_forward_hook(hook)
    try:
        model.eval()
        with torch.enable_grad():
            logits = model(x.unsqueeze(0).to(device))
            model.zero_grad(set_to_none=True)
            logits[0, class_idx].backward()
    finally:
        handle.remove()
    w = store["grad"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * store["act"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    cam = cam - cam.min()
    return (cam / cam.max().clamp_min(1e-8)).cpu().numpy()


test_ids = test_df.image_id.tolist()
wrong = np.flatnonzero(test_pred != test_labels)
wrong = wrong[np.argsort(-test_probs[wrong].max(1))][:10]  # most confident mistakes first

mistakes = pd.DataFrame({
    "image_id": [test_ids[i] for i in wrong],
    "true_class": [CLASSES[test_labels[i]] for i in wrong],
    "model_said": [CLASSES[test_pred[i]] for i in wrong],
    "confidence": [float(test_probs[i].max()) for i in wrong],
})
mistakes.to_csv(OUT_DIR / "mistakes.csv", index=False)

if len(wrong) == 0:
    print("No mistakes on the test set!")
else:
    cols, groups = min(5, len(wrong)), math.ceil(len(wrong) / 5)  # per 5 mistakes: photos on top, heatmaps below
    fig, axes = plt.subplots(groups * 2, cols, figsize=(4 * cols, 9.5 * groups), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for k, i in enumerate(wrong):
        crop = eval_view(ImageOps.exif_transpose(Image.open(DATA_DIR / "train_images" / test_ids[i])).convert("RGB"))
        cam = grad_cam(model, to_tensor(crop), int(test_pred[i]))
        top, bottom = axes[(k // 5) * 2, k % 5], axes[(k // 5) * 2 + 1, k % 5]
        top.imshow(crop)
        top.set_title(f"{k + 1}. {test_ids[i]}\ntrue: {CLASSES[test_labels[i]]}  ·  model: {CLASSES[test_pred[i]]} "
                      f"({test_probs[i].max():.0%})", fontsize=11)
        bottom.imshow(crop)
        bottom.imshow(cam, cmap="jet", alpha=0.45)
        bottom.set_title(f"where it looked for '{CLASSES[test_pred[i]]}'", fontsize=10)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "mistakes.png", dpi=100)
    plt.show()

    print("| # | photo | true class | model said | confidence | my reason |")
    print("|---|---|---|---|---:|---|")
    for k, row in enumerate(mistakes.itertuples(), 1):
        print(f"| {k} | {row.image_id} | {row.true_class} | {row.model_said} | {row.confidence:.0%} | ? |")

## 9. *(Optional)* Test on your own photos

The test set comes from the **same farms and cameras** as the training photos, so it is still a bit easy.
Real users will send different photos. To check how the model does on those:

1. Collect 50–100 cassava leaf photos that are **not** from this Kaggle dataset (for example your own phone photos).
2. Put them in folders named after the classes: `CBB/`, `CBSD/`, `CGM/`, `CMD/` and `Healthy/`.
3. Upload the parent folder as a Kaggle dataset (**Add Input → Upload**), then set `MY_PHOTOS_DIR` in section 1
   to its path, for example `/kaggle/input/my-cassava-photos`, and run again.

If the score drops a lot compared with the test set, that is an important finding: write about it in your README.

In [ ]:
if MY_PHOTOS_DIR:
    paths, labels = [], []
    for c, name in enumerate(CLASSES):
        for p in sorted(Path(MY_PHOTOS_DIR, name).glob("*")):
            if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
                paths.append(p)
                labels.append(c)
    if not paths:
        print(f"No photos found. Expected the folders {CLASSES} inside {MY_PHOTOS_DIR}")
    else:
        my_probs, my_labels, _ = evaluate(model, make_loader(paths, labels, eval_tf))
        results["my_photos"] = scores(my_labels, my_probs.argmax(1))
        print(f"{len(paths)} of your photos: accuracy {results['my_photos']['accuracy']:.3f}, "
              f"macro F1 {results['my_photos']['macro_f1']:.3f}")
        print(f"Kaggle test set:     accuracy {results['finetuned']['accuracy']:.3f}, "
              f"macro F1 {results['finetuned']['macro_f1']:.3f}\n")
        print(classification_report(my_labels, my_probs.argmax(1), labels=range(NUM_CLASSES),
                                    target_names=CLASSES, digits=3, zero_division=0))
else:
    print("Skipped: MY_PHOTOS_DIR is not set.")

## 10. Save everything

| File | What it is |
|---|---|
| `cassava_convnext_tiny.pt` | the fine-tuned model, with its class names and image size (for the app) |
| `results.json` | every score and the settings used |
| `class_counts.png`, `training_curves.png`, `confusion_matrix.png`, `mistakes.png` | charts for your README |
| `history.csv`, `split.csv`, `mistakes.csv` | the numbers behind the charts |
| `baseline.pt`, `finetuned.pt` | raw weights from the two steps (you can delete these) |

In [ ]:
torch.save(
    {
        "arch": "convnext_tiny",
        "state_dict": model.state_dict(),
        "classes": CLASSES,
        "long_names": LONG_NAMES,
        "img_size": IMG_SIZE,
        "resize_size": RESIZE_SIZE,
        "mean": MEAN,
        "std": STD,
    },
    OUT_DIR / "cassava_convnext_tiny.pt",
)

settings = dict(QUICK_RUN=QUICK_RUN, IMG_SIZE=IMG_SIZE, BATCH_SIZE=BATCH_SIZE, BASELINE_EPOCHS=BASELINE_EPOCHS,
                FINETUNE_EPOCHS=FINETUNE_EPOCHS, BASELINE_LR=BASELINE_LR, FINETUNE_LR=FINETUNE_LR,
                WEIGHT_DECAY=WEIGHT_DECAY, LABEL_SMOOTHING=LABEL_SMOOTHING, USE_CLASS_WEIGHTS=USE_CLASS_WEIGHTS,
                SEED=SEED)
summary = {
    "test": results,
    "photos": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "classes": dict(zip(CLASSES, LONG_NAMES)),
    "settings": settings,
    "torch": torch.__version__,
    "gpu": gpu_name,
    "training_minutes": round(float(hist.minutes.sum()), 1),
}
with open(OUT_DIR / "results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Results table for your README:\n")
print("| Model | Test accuracy | Test macro F1 |")
print("|---|---:|---:|")
labels_for_readme = {"always_" + CLASSES[majority]: f'Always answer "{CLASSES[majority]}"',
                     "baseline": "Baseline (last layer only)", "finetuned": "Fine-tuned (all layers)",
                     "my_photos": "Fine-tuned, on my own photos"}
for key, s in results.items():
    print(f"| {labels_for_readme[key]} | {s['accuracy']:.3f} | {s['macro_f1']:.3f} |")
print("\nSaved:", ", ".join(sorted(p.name for p in OUT_DIR.iterdir() if p.is_file())))

## What to do next

1. **Download the Output files.** Put the charts in your repo's `docs/` folder and the results table in the README.
2. **Fill in "my reason"** for the 10 mistakes. This shows you understand the model, not only that you ran it.
3. **Make this Kaggle notebook public** and link it from your README, so people can see the full run.
4. **Compare carefully:** the Kaggle winners got about 0.91 accuracy, but on a *hidden* test set, not your 10 %.
   Say this when you compare.
5. **Try to beat your own score**, changing **one thing at a time** and writing down the result:
   a bigger `IMG_SIZE` (448 or 512), more epochs, `USE_CLASS_WEIGHTS = False`, or a bigger model (ConvNeXt-Small).
6. **Add it to the app** as a second mode that loads `cassava_convnext_tiny.pt`. Keep the weights file out of git:
   upload it to the Hugging Face Hub instead (it is about 110 MB).